# Smart Industrial Maintenance System — Repo Pipeline
## FSE 570 Capstone | Arizona State University

This notebook runs the full **C-MAPSS turbofan engine** maintenance pipeline using **the repository's
modular source code** (`src/`, `config.py`). It imports all models, preprocessing,
explainability, optimization, and evaluation modules rather than re-implementing them inline.

> **Prerequisites:** Install the project in editable mode first:
> ```bash
> pip install -e .
> ```
> This makes `config`, `src.*`, etc. importable from anywhere.

| Module | Purpose |
|---|---|
| `config` | Central hyperparameters, paths, device config |
| `src.data.download` | C-MAPSS dataset download & loading (all 4 subsets) |
| `src.data.preprocess` | Normalization, splitting, synthetic augmentation |
| `src.data.synthetic_cmapss` | Synthetic degradation trajectory generation |
| `src.models.autoencoder` | LSTM Autoencoder (anomaly detection) |
| `src.models.lstm_predictor` | LSTM + Attention (failure classification) |
| `src.models.xgboost_rul` | XGBoost (RUL regression) |
| `src.models.bayesian_survival` | Weibull AFT (survival analysis) |
| `src.optimization.milp_scheduler` | MILP maintenance scheduling |
| `src.evaluation.simulation` | Monte Carlo policy comparison |
| `src.explainability.shap_analysis` | SHAP feature attribution |
| `src.explainability.attention_viz` | Temporal attention visualization |

---
## 1. Imports & Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
%matplotlib inline
import matplotlib.pyplot as plt

# --- Project modules (available via `pip install -e .`) ---
import config
from src.data.download import download_cmapss, load_cmapss_train, load_cmapss_all_subsets
from src.data.preprocess import DataPreprocessor
from src.data.synthetic_cmapss import SyntheticCMAPSSGenerator
from src.models.autoencoder import LSTMAutoencoder, AutoencoderTrainer
from src.models.lstm_predictor import LSTMPredictor, PredictorTrainer
from src.models.xgboost_rul import XGBoostRUL
from src.models.bayesian_survival import BayesianSurvival
from src.optimization.milp_scheduler import MaintenanceScheduler
from src.evaluation.simulation import MaintenanceSimulator
from src.explainability.shap_analysis import SHAPExplainer
from src.explainability.attention_viz import AttentionVisualizer

# Reproducibility
torch.manual_seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)

config.print_system_info()
print(f"Random Seed: {config.RANDOM_SEED}")

---
## 2. Download & Load C-MAPSS Data

In [ ]:
# Download C-MAPSS if not present
download_cmapss()

# Load all 4 subsets (FD001-FD004) with globally unique unit IDs
df_raw = load_cmapss_all_subsets()

print(f"\nLoaded {df_raw['unit_id'].nunique()} units, {len(df_raw)} rows")
print(f"Subsets: {config.CMAPSS_SUBSETS}")
print(f"Columns: {list(df_raw.columns[:10])}...")

---
## 3. Preprocessing & Synthetic Augmentation

The `DataPreprocessor` handles the full pipeline:
1. Drop constant / near-constant sensors (per `config.SENSORS_TO_DROP`)
2. Handle missing values (forward-fill per unit)
3. Temporal train/val/test split by unit ID (34 / 33 / 33 %)
4. Synthetic augmentation — injects ~30 % synthetic C-MAPSS trajectories into **training only**
5. Min-max normalization (fit on augmented training set, transform val/test)
6. Sliding window sequences (30-cycle windows) for LSTM input

In [ ]:
# Drop 'subset' column (not a feature)
df_for_preprocess = df_raw.drop(columns=["subset"], errors="ignore")

preprocessor = DataPreprocessor()
data = preprocessor.fit_transform(df_for_preprocess, augment=config.SYNTHETIC_AUGMENT)
preprocessor.save()

# Unpack splits
X_train = data["train"]["X"]
y_train_rul = data["train"]["y_rul"]
y_train_binary = data["train"]["y_binary"]
X_val = data["val"]["X"]
y_val_rul = data["val"]["y_rul"]
y_val_binary = data["val"]["y_binary"]
X_test = data["test"]["X"]
y_test_rul = data["test"]["y_rul"]
y_test_binary = data["test"]["y_binary"]

n_features = X_train.shape[2]

print(f"\n{'Split':<8} {'Sequences':>10} {'Failure%':>10} {'RUL range':>15}")
print("-" * 45)
for name, d in data.items():
    print(f"{name:<8} {d['X'].shape[0]:>10} {d['y_binary'].mean():>10.2%} "
          f"[{d['y_rul'].min():.0f}, {d['y_rul'].max():.0f}]")

---
## 4. Model 1: LSTM Autoencoder (Anomaly Detection)

Trained on **healthy data only** (RUL > failure horizon). Reconstruction error serves
as the anomaly score — degrading bearings produce patterns the autoencoder cannot
reconstruct well.

In [ ]:
autoencoder = LSTMAutoencoder(input_dim=n_features, seq_len=config.SEQUENCE_LENGTH)
ae_trainer = AutoencoderTrainer(autoencoder)

# Filter healthy samples for training (RUL > 50 % of MAX_RUL)
healthy_threshold = config.MAX_RUL * 0.5
healthy_mask = y_train_rul > healthy_threshold
X_healthy = X_train[healthy_mask]
X_val_ae = X_val[y_val_rul > healthy_threshold] if len(X_val) > 0 else None

if len(X_healthy) < 10:
    print(f"Warning: Only {len(X_healthy)} healthy samples. Using all training data.")
    X_healthy = X_train
    X_val_ae = X_val

print(f"Training autoencoder on {len(X_healthy)} healthy samples "
      f"(out of {len(X_train)} total, threshold RUL>{healthy_threshold:.0f})")

ae_trainer.train(X_healthy, X_val_ae if X_val_ae is not None and len(X_val_ae) > 0 else None)
ae_trainer.save_model(os.path.join(config.MODELS_DIR, "autoencoder.pt"))

In [ ]:
# Anomaly scores on healthy data for threshold
healthy_scores = autoencoder.compute_anomaly_score(torch.tensor(X_healthy, dtype=torch.float32))
autoencoder.set_threshold(healthy_scores)

# Anomaly scores on test set
anomaly_scores = autoencoder.compute_anomaly_score(torch.tensor(X_test, dtype=torch.float32))
_, anomalies = autoencoder.detect_anomalies(torch.tensor(X_test, dtype=torch.float32))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly score distribution
axes[0].hist(anomaly_scores, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(autoencoder.threshold, color="red", linestyle="--", label=f"Threshold = {autoencoder.threshold:.4f}")
axes[0].set_xlabel("Reconstruction Error (MSE)")
axes[0].set_ylabel("Count")
axes[0].set_title("Anomaly Score Distribution (Test Set)")
axes[0].legend()

# Anomaly score vs RUL
scatter = axes[1].scatter(y_test_rul, anomaly_scores, c=anomalies.astype(int),
                          cmap="coolwarm", alpha=0.6, s=20)
axes[1].axhline(autoencoder.threshold, color="red", linestyle="--", alpha=0.7)
axes[1].set_xlabel("Remaining Useful Life")
axes[1].set_ylabel("Anomaly Score")
axes[1].set_title("Anomaly Score vs RUL")
plt.colorbar(scatter, ax=axes[1], label="Anomaly Detected")

plt.tight_layout()
plt.show()

print(f"Anomalies detected: {anomalies.sum()}/{len(anomalies)} "
      f"({anomalies.mean():.1%})")

---
## 5. Model 2: LSTM Failure Predictor with Attention

Binary classifier predicting P(failure within h cycles). Uses an attention mechanism
to weight which time steps are most informative for the prediction.

In [ ]:
predictor = LSTMPredictor(input_dim=n_features)
pred_trainer = PredictorTrainer(predictor)
pred_trainer.train(X_train, y_train_binary, X_val, y_val_binary)
pred_trainer.save_model(os.path.join(config.MODELS_DIR, "lstm_predictor.pt"))

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, f1_score)

# Evaluate on test set
y_proba, _ = predictor.predict_proba(torch.tensor(X_test, dtype=torch.float32))
y_pred = (y_proba >= 0.5).astype(int)

print("Classification Report (Test Set):")
print(classification_report(y_test_binary.astype(int), y_pred,
                            target_names=["Normal", "Failure"], zero_division=0))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test_binary, y_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test_binary.astype(int), y_pred)
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Normal", "Failure"])
axes[0].set_yticklabels(["Normal", "Failure"])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=16)
plt.colorbar(im, ax=axes[0])

# ROC curve
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC = {roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Failure Predictor")
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"F1 Score: {f1_score(y_test_binary.astype(int), y_pred, zero_division=0):.4f}")
print(f"ROC AUC:  {roc_auc:.4f}")

---
## 6. Model 3: XGBoost RUL Estimation

Gradient-boosted regression on engineered tabular C-MAPSS features for remaining useful life prediction.
Uses the same unit-based split seed as the sequential preprocessor.

In [ ]:
from src.data.feature_engineering import FeatureEngineer

# Feature engineering on raw data for XGBoost
df_for_fe = df_raw.drop(columns=["subset"], errors="ignore").copy()
fe = FeatureEngineer()
df_engineered = fe.engineer_features(df_for_fe)

# Unit-based temporal split (same seed as preprocessor)
exclude_cols = ["unit_id", "cycle", "RUL"]
feature_cols = [c for c in df_engineered.columns if c not in exclude_cols]

unit_ids = df_engineered["unit_id"].unique()
np.random.seed(config.RANDOM_SEED)
np.random.shuffle(unit_ids)
n = len(unit_ids)
n_train = int(n * config.TRAIN_RATIO)
n_val = int(n * config.VAL_RATIO)
train_units = unit_ids[:n_train]
val_units = unit_ids[n_train:n_train + n_val]
test_units = unit_ids[n_train + n_val:]

X_train_xgb = df_engineered[df_engineered["unit_id"].isin(train_units)][feature_cols]
y_train_xgb = df_engineered[df_engineered["unit_id"].isin(train_units)]["RUL"]
X_val_xgb = df_engineered[df_engineered["unit_id"].isin(val_units)][feature_cols]
y_val_xgb = df_engineered[df_engineered["unit_id"].isin(val_units)]["RUL"]
X_test_xgb = df_engineered[df_engineered["unit_id"].isin(test_units)][feature_cols]
y_test_xgb = df_engineered[df_engineered["unit_id"].isin(test_units)]["RUL"]

xgb_model = XGBoostRUL()
xgb_model.train(X_train_xgb, y_train_xgb.values, X_val_xgb, y_val_xgb.values,
                feature_names=feature_cols)

print("\nTest Set Evaluation:")
xgb_model.evaluate(X_test_xgb, y_test_xgb.values)

In [ ]:
# Visualize XGBoost results
y_pred_rul = xgb_model.predict(X_test_xgb)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs Actual
axes[0].scatter(y_test_xgb, y_pred_rul, alpha=0.5, s=15, c="steelblue")
lims = [0, max(y_test_xgb.max(), y_pred_rul.max()) + 5]
axes[0].plot(lims, lims, "r--", lw=1.5, label="Perfect")
axes[0].set_xlabel("Actual RUL")
axes[0].set_ylabel("Predicted RUL")
axes[0].set_title("XGBoost: Predicted vs Actual RUL")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_pred_rul - y_test_xgb.values
axes[1].hist(residuals, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Prediction Error (Predicted - Actual)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")

# Feature importance (top 15)
if xgb_model.feature_importance is not None:
    fi = xgb_model.feature_importance.head(15)
    axes[2].barh(fi["feature"], fi["importance"], color="steelblue")
    axes[2].set_xlabel("Importance")
    axes[2].set_title("Top 15 Features (XGBoost)")
    axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

xgb_model.save(os.path.join(config.MODELS_DIR, "xgboost_model.pkl"))

---
## 7. Model 4: Bayesian Weibull Survival Analysis

Weibull Accelerated Failure Time (AFT) model for time-to-failure prediction with
uncertainty quantification. Uses active C-MAPSS sensors plus cycle count as covariates.

In [ ]:
# Prepare survival data using C-MAPSS active sensors + cycle
survival_features = config.ACTIVE_SENSORS + ["cycle"]
survival_cols = [c for c in survival_features if c in df_raw.columns] + ["RUL"]

df_survival_train = df_raw[df_raw["unit_id"].isin(train_units)][["unit_id"] + survival_cols].copy()
df_survival_test = df_raw[df_raw["unit_id"].isin(test_units)][["unit_id"] + survival_cols].copy()

survival_model = BayesianSurvival()

try:
    survival_model.fit(df_survival_train)
    survival_model.save(os.path.join(config.MODELS_DIR, "survival_model.pkl"))

    # Evaluate on test
    eval_results = survival_model.evaluate(df_survival_test)
    print(f"\nConcordance Index: {eval_results.get('concordance_index', 'N/A')}")

    # Predict with uncertainty
    predictions = survival_model.predict_with_uncertainty(df_survival_test)
    if predictions is not None:
        print(f"\nSample predictions (first 5):")
        print(predictions.head())

except Exception as e:
    print(f"Survival model fitting failed: {e}")
    print("Skipping — survival analysis may need more data points.")

---
## 8. Inference Pipeline

Combine all four models into a unified inference pass: anomaly detection,
failure probability, RUL estimate, and survival prediction.

In [ ]:
print("=" * 60)
print("INFERENCE PIPELINE — Test Set")
print("=" * 60)

# 1. Anomaly detection
anomaly_scores_all = autoencoder.compute_anomaly_score(torch.tensor(X_test, dtype=torch.float32))
_, is_anomaly = autoencoder.detect_anomalies(torch.tensor(X_test, dtype=torch.float32))

# 2. Failure probability
failure_proba, _ = predictor.predict_proba(torch.tensor(X_test, dtype=torch.float32))

# 3. XGBoost RUL
rul_pred = xgb_model.predict(X_test_xgb)

# Build results table
n_results = min(len(anomaly_scores_all), len(failure_proba), len(rul_pred))
results_df = pd.DataFrame({
    "anomaly_score": anomaly_scores_all[:n_results],
    "is_anomaly": is_anomaly[:n_results],
    "failure_prob": failure_proba[:n_results],
    "predicted_rul": rul_pred[:n_results],
    "actual_rul": y_test_rul[:n_results],
})

# Assign risk levels
def assign_risk(row):
    if row["failure_prob"] >= 0.7 or row["is_anomaly"]:
        return "CRITICAL"
    elif row["failure_prob"] >= 0.4:
        return "ELEVATED"
    return "NORMAL"

results_df["risk_level"] = results_df.apply(assign_risk, axis=1)

print(f"\nRisk Distribution:")
print(results_df["risk_level"].value_counts())
print(f"\nSample Predictions (first 10):")
print(results_df.head(10).to_string(index=False))

---
## 9. MILP Maintenance Scheduling

Formulates a Mixed-Integer Linear Program to optimally schedule maintenance jobs
given predicted failure risks, crew constraints, and cost parameters.

In [ ]:
# Generate machine risks from predictions (simulate a fleet of engines)
np.random.seed(config.RANDOM_SEED)
n_machines = 20
machine_risks = {}
for i in range(n_machines):
    idx = np.random.randint(0, len(failure_proba))
    machine_risks[i] = float(failure_proba[idx])

machine_names = {i: f"Engine-{i+1:02d}" for i in range(n_machines)}

scheduler = MaintenanceScheduler()
schedule_result = scheduler.create_schedule(
    machine_risks,
    n_time_slots=config.SCHEDULING_HORIZON,
    machine_names=machine_names,
)

print(f"\nOptimization Status: {schedule_result['status']}")
print(f"Total Estimated Cost: ${schedule_result['total_cost']:,.2f}")
print(f"\nSchedule:")
print(schedule_result['schedule'].to_string(index=False))

---
## 10. Monte Carlo Simulation — Policy Comparison

Simulates three maintenance policies over many runs to compare:
1. **Reactive** — fix after failure
2. **Scheduled** — fixed-interval preventive maintenance
3. **Optimized** — risk-based scheduling (our approach)

In [ ]:
simulator = MaintenanceSimulator(n_machines=20, n_periods=100)
sim_df, sim_summary = simulator.run_comparison(n_simulations=50)

print("\nSimulation Summary:")
for policy, metrics in sim_summary.items():
    print(f"\n  {policy}:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"    {k}: {v:,.2f}")
        else:
            print(f"    {k}: {v}")

In [ ]:
# Visualize simulation results
policies = sim_df["policy"].unique()
colors = {"reactive": "#FF4444", "scheduled": "#FFAA00", "optimized": "#44BB44"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for metric, ax, title in zip(
    ["total_cost", "total_downtime_hours", "availability_pct"],
    axes,
    ["Total Cost ($)", "Total Downtime (hrs)", "Availability (%)"]
):
    data_plot = [sim_df[sim_df["policy"] == p][metric].values for p in policies]
    bp = ax.boxplot(data_plot, tick_labels=[p.title() for p in policies], patch_artist=True)
    for patch, policy in zip(bp["boxes"], policies):
        patch.set_facecolor(colors.get(policy, "#888888"))
        patch.set_alpha(0.7)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.grid(True, alpha=0.3)

plt.suptitle("Monte Carlo Simulation: Maintenance Policy Comparison",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 11. Explainability

### 11a. SHAP Feature Attribution (XGBoost)
Shows which features drive the RUL predictions.

In [ ]:
shap_explainer = SHAPExplainer(xgb_model, model_type="xgboost")
shap_explainer.setup_explainer()
shap_values = shap_explainer.compute_shap_values(X_test_xgb)

# Sensor ranking
sensor_ranking = shap_explainer.get_sensor_ranking()

In [ ]:
# SHAP plots (inline)
import shap

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

plt.sca(axes[0])
shap.summary_plot(shap_values, shap_explainer.X_sample,
                  feature_names=shap_explainer.feature_names,
                  plot_type="bar", max_display=15, show=False)
axes[0].set_title("Global Feature Importance (SHAP)", fontsize=13, fontweight="bold")

plt.sca(axes[1])
shap.summary_plot(shap_values, shap_explainer.X_sample,
                  feature_names=shap_explainer.feature_names,
                  max_display=15, show=False)
axes[1].set_title("SHAP Beeswarm", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

### 11b. Attention Visualization (LSTM Predictor)

Shows which time steps the LSTM predictor attends to when making failure predictions.

In [ ]:
attn_viz = AttentionVisualizer(predictor)

# 1. Attention heatmap
attn_weights, attn_preds = attn_viz.extract_attention(X_test)

fig, ax = plt.subplots(figsize=(14, 6))
import seaborn as sns

n_show = min(15, len(attn_weights))
sns.heatmap(attn_weights[:n_show], cmap="YlOrRd", ax=ax,
            xticklabels=5,
            yticklabels=[f"Sample {i} (P={attn_preds[i]:.2f})" for i in range(n_show)],
            cbar_kws={"label": "Attention Weight"})
ax.set_xlabel("Time Step")
ax.set_ylabel("Sample")
ax.set_title("Temporal Attention Weights — LSTM Failure Predictor", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 2. Average attention: failure vs normal
fail_mask = y_test_binary == 1
normal_mask = y_test_binary == 0

avg_fail = attn_weights[fail_mask].mean(axis=0) if fail_mask.sum() > 0 else np.zeros(attn_weights.shape[1])
avg_normal = attn_weights[normal_mask].mean(axis=0) if normal_mask.sum() > 0 else np.zeros(attn_weights.shape[1])

fig, ax = plt.subplots(figsize=(12, 5))
x_axis = np.arange(len(avg_fail))
ax.fill_between(x_axis, avg_fail, alpha=0.3, color="#FF4444", label="Failure Cases")
ax.fill_between(x_axis, avg_normal, alpha=0.3, color="#44BB44", label="Normal Cases")
ax.plot(x_axis, avg_fail, color="#FF4444", linewidth=2)
ax.plot(x_axis, avg_normal, color="#44BB44", linewidth=2)
ax.set_xlabel("Time Step (Most Recent Snapshots)")
ax.set_ylabel("Average Attention Weight")
ax.set_title("Attention Pattern: Failure vs Normal", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 12. Overfitting Diagnostic

Compare train vs test metrics to check for data leakage or overfitting.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

print("=" * 60)
print("OVERFITTING DIAGNOSTIC")
print("=" * 60)

# LSTM Predictor: train vs test
y_proba_train, _ = predictor.predict_proba(torch.tensor(X_train, dtype=torch.float32))
y_pred_train = (y_proba_train >= 0.5).astype(int)
f1_train = f1_score(y_train_binary.astype(int), y_pred_train, zero_division=0)

y_proba_test, _ = predictor.predict_proba(torch.tensor(X_test, dtype=torch.float32))
y_pred_test = (y_proba_test >= 0.5).astype(int)
f1_test = f1_score(y_test_binary.astype(int), y_pred_test, zero_division=0)

print(f"\nLSTM Predictor:")
print(f"  Train F1: {f1_train:.4f}")
print(f"  Test  F1: {f1_test:.4f}")
print(f"  Gap:      {abs(f1_train - f1_test):.4f} {'(OK)' if abs(f1_train - f1_test) < 0.15 else '(POSSIBLE OVERFIT)'}")

# XGBoost: train vs test
y_rul_train_pred = xgb_model.predict(X_train_xgb)
y_rul_test_pred = xgb_model.predict(X_test_xgb)

mae_train = mean_absolute_error(y_train_xgb, y_rul_train_pred)
mae_test = mean_absolute_error(y_test_xgb, y_rul_test_pred)
r2_train = r2_score(y_train_xgb, y_rul_train_pred)
r2_test = r2_score(y_test_xgb, y_rul_test_pred)

print(f"\nXGBoost RUL:")
print(f"  Train MAE: {mae_train:.2f}  |  Test MAE: {mae_test:.2f}")
print(f"  Train R2:  {r2_train:.4f}  |  Test R2:  {r2_test:.4f}")
print(f"  MAE Gap:   {abs(mae_train - mae_test):.2f} {'(OK)' if abs(mae_train - mae_test) < 10 else '(POSSIBLE OVERFIT)'}")

# Autoencoder: healthy vs degraded
scores_healthy = autoencoder.compute_anomaly_score(torch.tensor(X_healthy[:100], dtype=torch.float32))
scores_test = autoencoder.compute_anomaly_score(torch.tensor(X_test, dtype=torch.float32))
print(f"\nAutoencoder Anomaly Scores:")
print(f"  Healthy mean: {scores_healthy.mean():.6f} +/- {scores_healthy.std():.6f}")
print(f"  Test mean:    {scores_test.mean():.6f} +/- {scores_test.std():.6f}")
print(f"  Separation:   {'GOOD' if scores_test.mean() > scores_healthy.mean() * 1.5 else 'WEAK'}")

---
## 13. Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Smart Industrial Maintenance — C-MAPSS Turbofan Analysis",
             fontsize=16, fontweight="bold")

# 1. Anomaly scores over time
axes[0, 0].scatter(range(len(anomaly_scores_all)), anomaly_scores_all,
                   c=is_anomaly.astype(int), cmap="coolwarm", s=10, alpha=0.6)
axes[0, 0].axhline(autoencoder.threshold, color="red", linestyle="--", alpha=0.7)
axes[0, 0].set_title("Anomaly Scores (Test)")
axes[0, 0].set_xlabel("Sample"); axes[0, 0].set_ylabel("Score")

# 2. Failure probability distribution
axes[0, 1].hist(failure_proba, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
axes[0, 1].axvline(0.5, color="red", linestyle="--", label="Threshold")
axes[0, 1].set_title("Failure Probability Distribution")
axes[0, 1].set_xlabel("P(Failure)"); axes[0, 1].legend()

# 3. RUL: predicted vs actual
axes[0, 2].scatter(y_test_xgb, y_pred_rul, alpha=0.5, s=15, c="steelblue")
axes[0, 2].plot([0, y_test_xgb.max()], [0, y_test_xgb.max()], "r--")
axes[0, 2].set_title("XGBoost RUL: Predicted vs Actual")
axes[0, 2].set_xlabel("Actual"); axes[0, 2].set_ylabel("Predicted")

# 4. Risk distribution (pie)
risk_counts = results_df["risk_level"].value_counts()
risk_colors = {"CRITICAL": "#FF4444", "ELEVATED": "#FFAA00", "NORMAL": "#44BB44"}
pie_colors = [risk_colors.get(r, "#888") for r in risk_counts.index]
axes[1, 0].pie(risk_counts.values, labels=risk_counts.index, colors=pie_colors,
               autopct="%1.1f%%", startangle=90)
axes[1, 0].set_title("Risk Level Distribution")

# 5. Attention heatmap (subset) — extract weights here
_, attn_weights = predictor.predict_proba(torch.tensor(X_test[:50], dtype=torch.float32))
n_attn = min(8, len(attn_weights))
import seaborn as sns
sns.heatmap(attn_weights[:n_attn], cmap="YlOrRd", ax=axes[1, 1],
            xticklabels=10, yticklabels=False, cbar_kws={"label": "Weight"})
axes[1, 1].set_title("Temporal Attention (Test Samples)")
axes[1, 1].set_xlabel("Time Step")

# 6. Simulation cost comparison
if not sim_summary.empty:
    policy_names = sim_summary.index.tolist()
    costs = sim_summary[('total_cost', 'mean')].values
    colors_map = {"reactive": "#FF4444", "scheduled": "#FFAA00", "optimized": "#44BB44"}
    bar_colors = [colors_map.get(p.lower().split()[0], "#888") for p in policy_names]
    axes[1, 2].bar([p.title() for p in policy_names], costs, color=bar_colors, alpha=0.8)
    axes[1, 2].set_title("Avg Cost by Policy")
    axes[1, 2].set_ylabel("Cost ($)")

plt.tight_layout()
plt.show()

---
## 14. Final Metrics Summary

In [ ]:
print("=" * 70)
print("  SMART INDUSTRIAL MAINTENANCE — FINAL RESULTS (C-MAPSS Turbofan)")
print("=" * 70)

print(f"\n  Device: {config.DEVICE}")
print(f"  Subsets: {config.CMAPSS_SUBSETS}")
print(f"  Units: {df_raw['unit_id'].nunique()}")
print(f"  Features: {n_features}")
print(f"  Sequence Length: {config.SEQUENCE_LENGTH}")
print(f"  Synthetic Augmentation: {config.SYNTHETIC_AUGMENT} "
      f"(target ratio: {config.SYNTHETIC_TARGET_RATIO:.0%})")

print(f"\n  {'Model':<30} {'Metric':<20} {'Value':>10}")
print(f"  {'-'*60}")
print(f"  {'LSTM Autoencoder':<30} {'Anomaly Rate':<20} {is_anomaly.mean():>10.2%}")
print(f"  {'LSTM Predictor':<30} {'Test F1':<20} {f1_test:>10.4f}")
print(f"  {'LSTM Predictor':<30} {'Test AUC':<20} {roc_auc:>10.4f}")
print(f"  {'XGBoost RUL':<30} {'Test MAE':<20} {mae_test:>10.2f}")
print(f"  {'XGBoost RUL':<30} {'Test R2':<20} {r2_test:>10.4f}")

if not sim_summary.empty:
    try:
        opt_cost = sim_summary.loc["Optimized (Risk-Based)", ('total_cost', 'mean')]
        react_cost = sim_summary.loc["Reactive", ('total_cost', 'mean')]
        if react_cost > 0:
            savings = (1 - opt_cost / react_cost) * 100
            print(f"  {'Optimized Policy':<30} {'Cost Savings vs Reactive':<20} {savings:>9.1f}%")
    except KeyError:
        pass

print(f"\n  Saved models:")
for f in sorted(os.listdir(config.MODELS_DIR)):
    if f.endswith((".pt", ".pkl")):
        size = os.path.getsize(os.path.join(config.MODELS_DIR, f)) / 1e6
        print(f"    {f} ({size:.2f} MB)")

print(f"\n{'='*70}")
print("  Pipeline complete. Run 'streamlit run dashboard/app.py' for the dashboard.")
print(f"{'='*70}")